# 02. Selección de variables con LASSO

Esta etapa aplica una regresión logística con penalización L1 para reducir la dimensionalidad del conjunto de datos. La selección se realiza sin utilizar sexo, raza o etnia como predictores; estas variables se reincorporan únicamente para poder auditar la equidad de los modelos posteriores.

**Entrada:** `data/processed/df_model.pkl`.

**Salida:** `data/processed/df_lasso2.csv`, con diez variables seleccionadas, tres atributos sensibles y la variable objetivo.


## 1. Dependencias y carga de datos


In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
import pandas as pd
import numpy as np
import os
import warnings


In [2]:
df_model = pd.read_pickle("data/processed/df_model.pkl")


## 2. Preprocesamiento y búsqueda del nivel de regularización


In [3]:
# ================================================================
# 0. IMPORTS
# ================================================================
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")


# ================================================================
# 1. CARGAR DATOS
# ================================================================

print("Shape original:", df_model.shape)
print("Columnas:", df_model.columns.tolist())


# ================================================================
# 2. FUNCIONES AUXILIARES
# ================================================================
def force_column_types(X, categorical_manual=None):
    X = X.copy()
    if categorical_manual:
        for col in categorical_manual:
            if col in X.columns:
                X[col] = X[col].astype('object')
    return X


def build_preprocessor(X):
    X = X.copy()
    
    categorical_manual = [
        'preaprobacion', 'tipo_prestamo', 'finalidad_prestamo',
        'prioridad_hipoteca', 'hipoteca_inversa', 'linea_credito_abierta',
        'finalidad_comercial', 'estado_hoepa', 'pago_global',
        'metodo_construccion', 'tipo_ocupacion', 'sistema_evaluacion_1',
        'motivo_denegacion', 'spread_faltante', 'sexo', 'raza', 'etnia'
    ]
    
    X = force_column_types(X, categorical_manual=categorical_manual)
    
    categorical_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    numeric_features = [c for c in X.columns if c not in categorical_features]
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('to_str', FunctionTransformer(lambda df: df.astype(str), feature_names_out='one-to-one')),
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    return preprocessor, numeric_features, categorical_features


# ================================================================
# 3. SEPARAR X / y Y QUITAR VARIABLES SENSIBLES
# ================================================================
TARGET = 'target'   #  cambia por el nombre real de tu target
SENSIBLES = ['sexo', 'raza', 'etnia']

y = df_model[TARGET]
X = df_model.drop(columns=[TARGET] + SENSIBLES)   # quitamos target y sensibles

print(f"\nVariables para LASSO: {X.shape[1]}")
print(X.columns.tolist())


# ================================================================
# 4. TRAIN / TEST SPLIT
# ================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")


# ================================================================
# 5. BARRIDO DE C PARA BUSCAR ~10 VARIABLES
# ================================================================
print(f"\n{'C':>12} {'Variables':>12}")
print("=" * 30)

Cs_probar = np.arange(0.000005, 0.0001, 0.000005)

for C in Cs_probar:
    preprocessor, _, _ = build_preprocessor(X_train)
    
    modelo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(
            penalty='l1', solver='saga', C=C,
            class_weight='balanced', max_iter=2000,
            random_state=42, n_jobs=-1
        ))
    ])
    
    modelo.fit(X_train, y_train)
    coefs = modelo.named_steps['model'].coef_[0]
    n_vars = (coefs != 0).sum()
    marca = " " if 9 <= n_vars <= 11 else ""
    print(f"{C:>12.6f} {n_vars:>12d}{marca}")


Shape original: (2743872, 25)
Columnas: ['anio_actividad', 'codigo_estado', 'tipo_producto_prestamo', 'categoria_vivienda', 'etnia', 'raza', 'sexo', 'preaprobacion', 'tipo_prestamo', 'finalidad_prestamo', 'importe_prestamo', 'ratio_prestamo_valor', 'valor_vivienda', 'tipo_ocupacion', 'ingresos', 'ratio_deuda_ingresos', 'etnia_co_solicitante', 'raza_co_solicitante', 'edad_solicitante', 'edad_co_solicitante', 'solicitante_mayor_62', 'sistema_evaluacion_1', 'ingreso_mediano_area', 'ingreso_relativo_zona', 'target']

Variables para LASSO: 21
['anio_actividad', 'codigo_estado', 'tipo_producto_prestamo', 'categoria_vivienda', 'preaprobacion', 'tipo_prestamo', 'finalidad_prestamo', 'importe_prestamo', 'ratio_prestamo_valor', 'valor_vivienda', 'tipo_ocupacion', 'ingresos', 'ratio_deuda_ingresos', 'etnia_co_solicitante', 'raza_co_solicitante', 'edad_solicitante', 'edad_co_solicitante', 'solicitante_mayor_62', 'sistema_evaluacion_1', 'ingreso_mediano_area', 'ingreso_relativo_zona']

X_train: (19

## 3. Ajuste del modelo LASSO definitivo


In [4]:
# ================================================================
# 6. ENTRENAR LASSO FINAL CON EL C ELEGIDO
# ================================================================
C_elegido = 0.000040   #  pon el que te dio ~10 variables

preprocessor, num_features, cat_features = build_preprocessor(X_train)

modelo_lasso = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        penalty='l1', solver='saga', C=C_elegido,
        class_weight='balanced', max_iter=2000,
        random_state=42, n_jobs=-1
    ))
])

modelo_lasso.fit(X_train, y_train)

feature_names = modelo_lasso.named_steps['preprocessor'].get_feature_names_out()
feature_names = [f.split('__', 1)[-1] for f in feature_names]

coefs = pd.Series(modelo_lasso.named_steps['model'].coef_[0], index=feature_names)
vars_supervivientes = coefs[coefs != 0].abs().sort_values(ascending=False).index.tolist()

print(f"\n{'='*60}")
print(f"Con C={C_elegido}: LASSO selecciona {len(vars_supervivientes)} de {len(feature_names)}")
print(f"{'='*60}")
for v in vars_supervivientes:
    tipo = "CAT" if v in cat_features else "NUM"
    print(f"  [{tipo}] {v:35s} coef: {coefs[v]:+.4f}")

# AUC sanity check
y_prob = modelo_lasso.predict_proba(X_train)[:, 1]
print(f"\nAUC train: {roc_auc_score(y_train, y_prob):.4f}")

y_prob_test = modelo_lasso.predict_proba(X_test)[:, 1]
print(f"AUC test:  {roc_auc_score(y_test, y_prob_test):.4f}")



Con C=4e-05: LASSO selecciona 10 de 21
  [NUM] ratio_deuda_ingresos                coef: -0.6392
  [NUM] importe_prestamo                    coef: +0.4570
  [CAT] sistema_evaluacion_1                coef: -0.4567
  [NUM] ingreso_mediano_area                coef: +0.0680
  [CAT] finalidad_prestamo                  coef: -0.0611
  [NUM] ingresos                            coef: +0.0571
  [NUM] ingreso_relativo_zona               coef: +0.0488
  [NUM] valor_vivienda                      coef: +0.0162
  [NUM] ratio_prestamo_valor                coef: -0.0134
  [CAT] codigo_estado                       coef: -0.0002

AUC train: 0.8566
AUC test:  0.8571


## 4. Construcción del conjunto reducido


In [5]:
# ================================================================
# 7. CREAR DATAFRAME FINAL: 10 VARS LASSO + 3 SENSIBLES + TARGET
# ================================================================
df_lasso2 = df_model[vars_supervivientes + SENSIBLES + [TARGET]].copy()

print(df_lasso2.shape)   # (N, 14) → 10 + 3 + 1
df_lasso2.head()

# Guardar
df_lasso2.to_csv('data/processed/df_lasso2.csv', index=False)
print(" Guardado df_lasso2.csv")


(2743872, 14)
 Guardado df_lasso2.csv
